## Notebook to demo creating a feature class from a spark dataframe.

Note: Execute this notebook in a Pro for a better experience.

References:
- https://pro.arcgis.com/en/pro-app/latest/arcpy/data-access/tabletoarrowtable.htm

In [ ]:
import os

# import geofunctions as S
# import arcpy
import pyarrow as pa

# import spark_esri as se
import pyspark.sql.functions as F

### Start a spark instance.

In [ ]:
extra_java_options = "-XX:+UseCompressedOops -XX:+AggressiveHeap"

config = {
    "spark.driver.memory": "31G",
    "spark.executor.memory": "31G",
    "spark.driver.extraJavaOptions": extra_java_options,
    "spark.executor.extraJavaOptions": extra_java_options,
    "spark.kryo.unsafe": True,
    "spark.memory.offHeap.enable": True,
    "spark.memory.offHeap.size": "31G",
    "spark.jars": "geofunctions-0.6.jar",
}

spark = se.spark_start(config=config)

### Create random data in WM SR in spark and convert it to Pandas.

In [ ]:
xmin, ymin, xmax, ymax = (
    45.89223985322122,
    28.236525532194733,
    49.216965081318975,
    30.514823980851915,
)
xdel = xmax - xmin
ydel = ymax - ymin

In [ ]:
pdf = (
    spark.range(100_000)
    .withColumn("x", S.st_lontox(F.rand() * xdel + xmin))
    .withColumn("y", S.st_lattoy(F.rand() * ydel + ymin))
    .withColumn("SHAPE", S.st_point("x", "y"))
    .drop("id")
    .toPandas()
)

In [ ]:
pdf = (
    spark.range(1_000)
    .withColumn("x", F.rand() * 360 - 180)
    .withColumn("y", F.rand() * 180 - 90)
    .withColumn(
        "SHAPE",
        F.concat(F.lit("POINT("), F.col("x"), F.lit(" "), F.col("y"), F.lit(")")),
    )
    # .show(truncate=False)
    .drop("id")
    .toPandas()
)

### Create Arrow schema and add SHAPE metadata.

In [ ]:
sp_ref = arcpy.SpatialReference(3857)
metadata = {"esri.encoding": "WKB", "esri.sr_wkt": sp_ref.exportToString()}
pa_shape = pa.field("SHAPE", pa.binary(), nullable=False, metadata=metadata)

schema = pa.Schema.from_pandas(pdf)
shape_index = schema.get_field_index("SHAPE")
schema = schema.set(shape_index, pa_shape)

### Create a feature class from Arrow table.

In [ ]:
tab = pa.Table.from_pandas(pdf, schema=schema)
ws = "memory"  # arcpy.env.scratchGDB
nm = "SparkPoints"
fc = os.path.join(ws, nm)
arcpy.management.Delete(fc)
arcpy.management.CopyFeatures(tab, fc)